In [ ]:
## Splitting images and masks into tiles / patches

In [ ]:
from rasterio.windows import Window

def process_full_scene(scene_path, model):
    with rasterio.open(scene_path) as src:
        width, height = src.width, src.height
        output = np.zeros((height, width, 3), dtype=np.float32)
        counts = np.zeros((height, width), dtype=np.uint8)
        
        # 1024x1024 sliding window (896 stride = 12.5% overlap)
        for y in range(0, height, 896):
            for x in range(0, width, 896):
                window = Window(x, y, 1024, 1024)
                patch = src.read(window=window)
                patch = preprocess(patch)  # Normalize
                pred = model.predict(patch[np.newaxis, ...])
                
                # Weighted blending in overlap areas
                blend_weights = np.hanning(1024)[:, None] * np.hanning(1024)
                output[y:y+1024, x:x+1024] += pred[0] * blend_weights
                counts[y:y+1024, x:x+1024] += blend_weights
        
        # Final prediction
        return output / counts[..., None]  # Normalize by weight sums

In [ ]:
import zarr
import numpy as np
import rasterio
from tqdm import tqdm
import os
from multiprocessing import Pool

class ZarrPatchStore:
    def __init__(self, output_path, patch_size=1024, overlap=128, chunk_size=32):
        """
        Initialize Zarr storage for patches
        
        Args:
            output_path: Directory to store Zarr datasets
            patch_size: Size of square patches (default: 1024)
            overlap: Overlap between patches (default: 128)
            chunk_size: Zarr chunk size (default: 32)
        """
        self.patch_size = patch_size
        self.stride = patch_size - overlap
        self.chunk_size = chunk_size
        self.store = zarr.DirectoryStore(output_path)
        self.root = zarr.group(store=self.store, overwrite=True)
        
    def process_scene(self, scene_path, band_order=['B2', 'B3', 'B4']):
        """Convert a scene into patches stored in Zarr format"""
        with rasterio.open(scene_path) as src:
            # Read all bands and stack them
            img = np.stack([src.read(i+1) for i in range(len(band_order))], axis=-1)
            height, width = img.shape[:2]
            
            # Create Zarr arrays
            self.root.create_dataset('patches',
                                   shape=(0, self.patch_size, self.patch_size, 3),
                                   chunks=(self.chunk_size,)*4,
                                   dtype='float32',
                                   compressor=zarr.Blosc(cname='zstd', clevel=3))
            
            self.root.create_dataset('coordinates',
                                   shape=(0, 2),
                                   dtype='int32')
            
            # Process patches in parallel
            coords = []
            for y in range(0, height - self.patch_size + 1, self.stride):
                for x in range(0, width - self.patch_size + 1, self.stride):
                    coords.append((y, x))
            
            with Pool() as pool:
                patches = list(tqdm(pool.imap(
                    lambda c: self._extract_patch(img, c),
                    coords,
                    total=len(coords)
                ))
            
            # Store in Zarr
            self.root['patches'].append(np.stack([p[0] for p in patches]))
            self.root['coordinates'].append(np.array([p[1] for p in patches]))
            
        # Add metadata
        self.root.attrs['patch_size'] = self.patch_size
        self.root.attrs['stride'] = self.stride
        self.root.attrs['band_order'] = band_order
        self.root.attrs['scene_path'] = scene_path
        
    def _extract_patch(self, img, coord):
        """Extract and preprocess a single patch"""
        y, x = coord
        patch = img[y:y+self.patch_size, x:x+self.patch_size]
        
        # Normalize to [0,1] using scene statistics
        patch = (patch - np.min(patch)) / (np.max(patch) - np.min(patch) + 1e-6)
        
        return patch, coord

    def get_patch_generator(self, batch_size=32, shuffle=True):
        """Create a TensorFlow data generator from Zarr store"""
        import tensorflow as tf
        
        def generator():
            indices = np.arange(len(self.root['patches']))
            if shuffle:
                np.random.shuffle(indices)
                
            for i in indices:
                yield (self.root['patches'][i],
                       self.root['coordinates'][i])
        
        return tf.data.Dataset.from_generator(
            generator,
            output_signature=(
                tf.TensorSpec(shape=(self.patch_size, self.patch_size, 3), dtype=tf.float32),
                tf.TensorSpec(shape=(2,), dtype=tf.int32)
            )
        ).batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
# 1. Initialize patch storage
patch_store = ZarrPatchStore(
    output_path='./patches.zarr',
    patch_size=1024,
    overlap=128
)

# 2. Process multiple scenes
scenes = ['scene1.tif', 'scene2.tif', 'scene3.tif']
for scene in scenes:
    patch_store.process_scene(scene)

# 3. Create TensorFlow dataset
train_dataset = patch_store.get_patch_generator(batch_size=32)

# 4. Train model
model.fit(train_dataset, epochs=50)

# 5. Access patches directly (alternative)
patches = zarr.open('./patches.zarr', mode='r')
print(f"Total patches: {len(patches['patches'])}")
print(f"Patch shape: {patches['patches'][0].shape}")
print(f"First patch coordinates: {patches['coordinates'][0]}")